# Foraging: every model, side by side

One report over the whole steering ladder on the `foraging` task -- from a policy given
nothing to one given the entire swimming circuit plus a turn primitive. For each model:
a physics-only success score, a training curve where it was trained, and a video + head
path.

## The models

| # | model | what it's given | steering source | learned? |
|---|---|---|---|---|
| 1 | `mlp` | nothing | a plain MLP reads the whole obs | swimming **and** steering |
| 2 | `ncap_blind` | the swimming circuit, **no path to the food** | none -- control condition | nothing steers |
| 3 | `ncap_insteer` | circuit + an in-circuit target projection | 8 sign-free weights in the circuit | the turn direction |
| 4 | `ncap_reflex` | circuit + a fixed P+D reflex | hand-derived reflex (no params) | only the circuit's turn gain |
| 5 | `ncap_hardcoded` | circuit + a fixed proportional turn | `steer_to_food`, **no training at all** | nothing |
| 6 | `ncap_mlp_steer` | circuit + a turn primitive | a tiny MLP on the bearing, warm-started | only *which way* to turn |

`ncap_blind` is the control: with `controller=None` and no in-circuit steering, nothing
carries the food's position into the policy, so it cannot navigate however long it trains.
It is here to show exactly that -- not as a claim that NCAP cannot steer (models 3-6 all
steer).

## What's measured

**Physics-only success** -- the fraction of fresh episodes whose head comes within
`SUCCESS_DIST` of the food, read from simulator state. Aggregate training reward is
unreliable in this environment (a high score once turned out to be pure swim speed with
no food-seeking under it), so it is used only for the "did it climb" curves.

In [ ]:
# --- Setup -------------------------------------------------------------------------
%matplotlib inline
import sys
from pathlib import Path

SRC = str(Path.cwd().parent / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from macrocircuits import ensure_tonic
ensure_tonic()

import torch, tonic, tonic.torch
from dm_control.mujoco import engine
from IPython.display import Video

from macrocircuits.models import ppo_swimmer_model
from macrocircuits.controllers import make_controller
from macrocircuits.video import write_video

N_JOINTS      = 5
SUCCESS_DIST  = 0.15
EVAL_SEED     = 0
EVAL_EPISODES = 30
FORAGE = 'data/local/experiments/tonic/swimmer-foraging'   # where the PPO runs live

# One entry per model. `path` is a trained checkpoint dir (None => build fresh/untrained,
# which is exactly right for the hardcoded reflex that needs no training). `controller`
# and `swimmer_kwargs` reproduce how the run was built, so a fresh actor matches its
# trained sibling's architecture.
MODELS = {
    'mlp': dict(
        label='MLP\n(nothing given)', color='#8a8a83',
        network='mlp', controller=None, swimmer_kwargs=None,
        path=f'{FORAGE}/mlp_ppo_forage_poc'),
    'ncap_blind': dict(
        label='NCAP\n(no steering wired)', color='#b0b0a8',
        network='ncap', controller=None, swimmer_kwargs=None,
        path=f'{FORAGE}/ncap_ppo_forage_poc'),
    'ncap_insteer': dict(
        label='NCAP + in-circuit\nsteering', color='#eda100',
        network='ncap', controller=None,
        swimmer_kwargs=dict(include_target_steering=True, n_turn_joints=2),
        path=f'{FORAGE}/ncap_insteer_ppo_forage_poc'),
    'ncap_reflex': dict(
        label='NCAP + fixed\nP+D reflex', color='#eb6834',
        network='ncap', controller='foraging',
        swimmer_kwargs=dict(include_speed_control=True),
        path=f'{FORAGE}/ncap_reflex_best'),
    'ncap_hardcoded': dict(
        label='NCAP + hardcoded\nsteer_to_food', color='#4a3aa7',
        network='ncap', controller='steer_to_food',
        swimmer_kwargs=dict(include_speed_control=True),
        path=None),   # no training -- fixed reflex, evaluated as-is
    'ncap_mlp_steer': dict(
        label='NCAP + learned\nMLP steering', color='#2a78d6',
        network='ncap', controller='learned_steering', swimmer_kwargs=None,
        path=f'{FORAGE}/ncap_steer_ppo_forage_poc'),
}
print(f'{len(MODELS)} models defined')

## Helpers

`load_model` returns the same `(policy, env, physics)` triple for every model, whether it
comes from a trained checkpoint or a freshly built (untrained) actor, so one `success_rate`
and one video path score them all. A missing checkpoint falls back to a fresh actor and is
flagged, so the report still renders end to end.

In [ ]:
def _get_physics(env):
    while env is not None:
        if hasattr(env, 'physics'):
            return env.physics
        inner = getattr(env, 'environment', None)
        if inner is not None and hasattr(inner, 'physics'):
            return inner.physics
        env = getattr(env, 'env', None)
    raise RuntimeError('no physics in env chain')


def _fresh_actor(spec, env, seed=EVAL_SEED):
    """Build an untrained actor for `spec` and initialize it against `env`. Used for the
    hardcoded model (no training) and as the fallback when a checkpoint is missing."""
    torch.manual_seed(seed)
    controller = make_controller(spec['controller'], N_JOINTS)
    model = ppo_swimmer_model(n_joints=N_JOINTS, critic_sizes=(256, 256), action_noise=0.1,
                              controller=controller, **(spec['swimmer_kwargs'] or {}))
    actor = model.actor
    actor.initialize(observation_space=env.observation_space, action_space=env.action_space)
    def policy(obs):
        with torch.no_grad():
            out = actor(torch.as_tensor(obs, dtype=torch.float32))
            return (out.loc if hasattr(out, 'loc') else out).numpy()
    return policy


def load_model(key, seed=EVAL_SEED):
    """(policy, env, physics, status) for a model. status is 'trained', 'untrained' (the
    hardcoded reflex), or 'MISSING -> untrained' (checkpoint absent, fell back)."""
    spec = MODELS[key]
    # Every model here is an RL/hardcoded actor that reads the timestep from the obs, so
    # they all evaluate on the same time-feature foraging env; no per-model reset needed.
    tf = spec['network'] == 'ncap'
    env = tonic.environments.distribute(
        lambda: tonic.environments.ControlSuite('swimmer-foraging', time_feature=tf))
    env.initialize(seed)

    ckpt = None
    if spec['path']:
        cp = os.path.join(spec['path'], 'checkpoints')
        if os.path.isdir(cp):
            ids = [int(n.split('.')[0][5:]) for n in os.listdir(cp) if n.startswith('step_')]
            if ids:
                ckpt = os.path.join(cp, f'step_{max(ids)}')

    if ckpt is None:
        status = 'untrained' if spec['path'] is None else 'MISSING -> untrained'
        return _fresh_actor(spec, env, seed), env, _get_physics(env.environments[0]), status

    # Rebuild the exact trained agent + env from the run's own config.yaml.
    import argparse, yaml
    import macrocircuits.training as _t
    ns = dict(vars(_t))
    cfg = argparse.Namespace(
        **yaml.load(open(os.path.join(spec['path'], 'config.yaml')), Loader=yaml.FullLoader))
    try:
        if cfg.header: exec(cfg.header, ns)
    except Exception:
        pass
    agent = eval(cfg.agent, ns)
    tenv = tonic.environments.distribute(lambda: eval(cfg.environment, ns))
    tenv.initialize(seed)
    agent.initialize(observation_space=tenv.observation_space,
                     action_space=tenv.action_space, seed=seed)
    agent.load(ckpt)
    return (lambda obs: agent.test_step(obs, 0)), tenv, _get_physics(tenv.environments[0]), 'trained'


def success_rate(policy, env, n_episodes=EVAL_EPISODES):
    """Fraction of fresh episodes whose head comes within SUCCESS_DIST of the food.
    Distance is read from the pre-reset transition obs, never a just-reset env."""
    tgt = slice(N_JOINTS, N_JOINTS + 2)
    obs = env.start(); mind = np.inf; hits = []
    while len(hits) < n_episodes:
        obs, infos = env.step(policy(obs))
        mind = min(mind, float(np.linalg.norm(infos['observations'][0, tgt])))
        if infos['resets'][0]:
            hits.append(mind < SUCCESS_DIST); mind = np.inf
    return float(np.mean(hits))


def training_curve(path):
    """(x_steps, y_reward) from a run's log.csv, or None if it has no log."""
    log_path = os.path.join(path, 'log.csv') if path else None
    if not log_path or not os.path.isfile(log_path):
        return None
    log = pd.read_csv(log_path)
    ycol = next(c for c in log.columns if 'episode_score/mean' in c)
    xcol = next(c for c in log.columns if c.endswith('/steps'))
    return log[xcol], log[ycol]


def _arena_frame(physics, distance=5.5):
    """Fixed overhead camera on the whole arena. Not a tracking camera: 0/1 follow the
    worm but never rotate with its heading, so a turn can read backwards; nothing rotates
    here, so direction on screen is real."""
    cam = engine.Camera(physics, height=480, width=640, camera_id=-1)
    rc = cam._render_camera
    rc.lookat[:] = [0.0, 0.0, 0.0]
    rc.distance = distance; rc.azimuth = 90; rc.elevation = -90
    frame = cam.render(); cam._scene.free()
    return frame


def forage_episode(policy, env, physics, steps=1000, viz_food_size=0.10):
    """One episode; records overhead frames, head path, food positions and the steps a
    food was reached (it then respawns)."""
    obs = env.start()
    if viz_food_size:
        physics.named.model.geom_size['target', 0] = viz_food_size; physics.forward()
    frames, hx, hy, fx, fy, eats = [], [], [], [], [], []
    prev = None
    for i in range(steps):
        frames.append(_arena_frame(physics))
        nose = physics.named.data.geom_xpos['nose'][:2].copy()
        food = physics.named.data.geom_xpos['target'][:2].copy()
        hx.append(nose[0]); hy.append(nose[1]); fx.append(food[0]); fy.append(food[1])
        if prev is not None and np.linalg.norm(food - prev) > 1e-6:
            eats.append(i)
        prev = food
        obs, infos = env.step(policy(obs))
        if infos['resets'][0]:
            break
    return dict(frames=frames, hx=np.array(hx), hy=np.array(hy),
                fx=np.array(fx), fy=np.array(fy), eats=eats)


def show_video(frames, path, fps=30):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    write_video(path, frames, fps=fps)
    return Video(path, embed=True, width=520)


def plot_path(ep, title, ax=None):
    ax = ax or plt.figure(figsize=(5.2, 5.2)).gca()
    ax.scatter(ep['hx'], ep['hy'], c=np.arange(len(ep['hx'])), cmap='viridis', s=5)
    food = np.unique(np.c_[ep['fx'], ep['fy']], axis=0)
    ax.scatter(food[:, 0], food[:, 1], marker='*', s=200, c='#e34948', ec='k', zorder=5)
    if ep['eats']:
        ax.scatter(ep['hx'][ep['eats']], ep['hy'][ep['eats']], s=70, c='#1baf7a', ec='k',
                   zorder=6, label='reached')
    ax.set_aspect('equal'); ax.grid(alpha=0.3); ax.set_title(title, fontsize=10)
    ax.set_xlabel('world x'); ax.set_ylabel('world y')


print('helpers ready')

## 1. Success -- the headline comparison

Every model on one axis, physics-only, sorted from what's given least to given most.

In [ ]:
RESULTS = {}
for key in MODELS:
    pol, env, _, status = load_model(key)
    RESULTS[key] = dict(success=success_rate(pol, env), status=status)
    print(f'{key:16} {status:22} {RESULTS[key]["success"]*100:3.0f}%')

# Table view alongside the bars, so the numbers exist as text and not only as bar height.
tbl = pd.DataFrame(
    [(MODELS[k]['label'].replace('\n', ' '), f'{RESULTS[k]["success"]*100:.0f}%',
      RESULTS[k]['status']) for k in MODELS],
    columns=['model', 'success', 'checkpoint'])
print('\n' + tbl.to_string(index=False))

In [ ]:
keys = list(MODELS)
vals = [RESULTS[k]['success'] * 100 for k in keys]
colors = [MODELS[k]['color'] for k in keys]

fig, ax = plt.subplots(figsize=(10, 4.8))
bars = ax.bar(range(len(keys)), vals, color=colors, edgecolor='white', linewidth=2, zorder=3)
ax.bar_label(bars, fmt='%.0f%%', padding=3, fontsize=11, color='#3d3d38')
ax.set_xticks(range(len(keys)), [MODELS[k]['label'] for k in keys], fontsize=9)
ax.set_ylabel('physics-only success (%)'); ax.set_ylim(0, 100)
ax.set_title(f'reaching food, {EVAL_EPISODES} fresh episodes (eval seed {EVAL_SEED})')
ax.grid(alpha=0.25, axis='y', zorder=0)
ax.spines[['top', 'right']].set_visible(False)
# Mark the models that were never trained, so a tall untrained bar can't mislead.
for i, k in enumerate(keys):
    if RESULTS[k]['status'] != 'trained':
        ax.text(i, 3, RESULTS[k]['status'].split()[0], ha='center', va='bottom',
                fontsize=7, color='#555', rotation=90)
plt.tight_layout(); plt.show()

## 2. Training curves

Reward vs environment steps, for the models that were trained (the hardcoded reflex has
none). Reward only -- "did it climb", not "did it forage". Read the bars above for that.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
any_curve = False
for k in MODELS:
    c = training_curve(MODELS[k]['path'])
    if c is None:
        continue
    ax.plot(c[0], c[1], marker='o', ms=3, lw=2, color=MODELS[k]['color'],
            label=MODELS[k]['label'].replace('\n', ' '))
    any_curve = True
if any_curve:
    ax.set_xlabel('environment steps'); ax.set_ylabel('test episode score (mean)')
    ax.legend(frameon=False, fontsize=8); ax.grid(alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_title('training reward (NOT a foraging measure)')
    plt.tight_layout(); plt.show()
else:
    print('no training logs found yet')

## 3. Trajectories

One episode per model, same eval seed: the head path (colored by time) and the food. A
model that forages sweeps toward the stars; a blind one wanders or circles.

In [ ]:
VIDEO_SEED = 3
EPISODES = {}
n = len(MODELS)
cols = 3
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4.6 * cols, 4.6 * rows))
axes = np.array(axes).reshape(-1)
for ax, key in zip(axes, MODELS):
    pol, env, phys, _ = load_model(key, seed=VIDEO_SEED)
    ep = forage_episode(pol, env, phys, steps=1000)
    EPISODES[key] = ep
    plot_path(ep, f'{MODELS[key]["label"].replace(chr(10), " ")}\nfood reached: {len(ep["eats"])}',
              ax=ax)
for ax in axes[n:]:
    ax.set_visible(False)
plt.tight_layout(); plt.show()

## 4. Videos

Overhead replay of the same episodes. The two best foragers first; the rest below.
Re-run a single cell to watch any one model.

In [ ]:
k = 'ncap_hardcoded'
print(MODELS[k]['label'].replace(chr(10), ' '),
      ' -- food reached:', len(EPISODES[k]['eats']))
show_video(EPISODES[k]['frames'][::2], f'output_videos/report_{k}.mp4', fps=30)

In [ ]:
k = 'ncap_mlp_steer'
print(MODELS[k]['label'].replace(chr(10), ' '),
      ' -- food reached:', len(EPISODES[k]['eats']))
show_video(EPISODES[k]['frames'][::2], f'output_videos/report_{k}.mp4', fps=30)

In [ ]:
k = 'ncap_reflex'
print(MODELS[k]['label'].replace(chr(10), ' '),
      ' -- food reached:', len(EPISODES[k]['eats']))
show_video(EPISODES[k]['frames'][::2], f'output_videos/report_{k}.mp4', fps=30)

In [ ]:
k = 'ncap_insteer'
print(MODELS[k]['label'].replace(chr(10), ' '),
      ' -- food reached:', len(EPISODES[k]['eats']))
show_video(EPISODES[k]['frames'][::2], f'output_videos/report_{k}.mp4', fps=30)

In [ ]:
k = 'ncap_blind'
print(MODELS[k]['label'].replace(chr(10), ' '),
      ' -- food reached:', len(EPISODES[k]['eats']))
show_video(EPISODES[k]['frames'][::2], f'output_videos/report_{k}.mp4', fps=30)

In [ ]:
k = 'mlp'
print(MODELS[k]['label'].replace(chr(10), ' '),
      ' -- food reached:', len(EPISODES[k]['eats']))
show_video(EPISODES[k]['frames'][::2], f'output_videos/report_{k}.mp4', fps=30)

## Notes & caveats

- **Single eval seed, 30 episodes.** Numbers shift ~±10% across eval seeds; differences
  under ~4 episodes are not distinguishable from noise. For a firm claim, average several
  seeds.
- **`ncap_hardcoded` is untrained** -- a fixed proportional turn toward the food. That it
  scores near the top with no learning at all is the point: once the geometry is right,
  navigating to food needs almost no learning.
- **`ncap_reflex`** is the config that reaches ~47% on the `improve-foraging-reflex`
  branch (fixed P+D reflex + `alignment_reward_weight`), reproduced here on the corrected
  axis convention.
- **`ncap_blind` is the control**, not a failed model: with no steering path wired in, no
  amount of training can help. Models 3-6 all steer, so this says nothing about NCAP's
  capacity -- only that a policy needs *some* path from the food's position to its actions.